In [11]:
import pandas as pd
import json
import os
from typing import Tuple, Dict, Any
import logging

In [12]:
# Configuración básica de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [13]:
# Constantes
DATA_PATH = os.path.join('..', 'data')
RAW_DATA_PATH = os.path.join(DATA_PATH, 'raw')
REPORTS_PATH = os.path.join('..', 'docs', 'reports')

In [14]:
def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza un DataFrame utilizando Min-Max Scaling.
    
    Args:
        df (pd.DataFrame): DataFrame a normalizar.
        
    Returns:
        pd.DataFrame: DataFrame normalizado.
        
    Raises:
        ValueError: Si el DataFrame tiene columnas con valores constantes.
    """
    if (df.max() - df.min()).eq(0).any():
        raise ValueError("Columnas con valores constantes no se pueden normalizar")
    return (df - df.min()) / (df.max() - df.min())

In [15]:
def standardize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Estandariza un DataFrame utilizando Z-Score.
    
    Args:
        df (pd.DataFrame): DataFrame a estandarizar.
        
    Returns:
        pd.DataFrame: DataFrame estandarizado.
    """
    return (df - df.mean()) / df.std()

In [16]:
def clean_dataframe(
    df: pd.DataFrame, 
    normalization_limit: int
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Limpia y transforma un DataFrame aplicando normalización y estandarización.
    
    Args:
        df (pd.DataFrame): DataFrame de entrada.
        normalization_limit (int): Índice de columna para dividir las operaciones.
        
    Returns:
        Tuple: DataFrame procesado y reporte de métricas.
        
    Raises:
        TypeError: Si los inputs son inválidos.
        ValueError: Si el límite está fuera de rango.
    """
    # Validación de inputs
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input debe ser un DataFrame")
    if not isinstance(normalization_limit, int) or normalization_limit < 0:
        raise ValueError("Límite debe ser un entero positivo")
    if normalization_limit > df.shape[1]:
        raise ValueError("Límite excede el número de columnas")

    try:
        df_clean = df.copy().drop_duplicates(keep=False)
        
        # Normalización
        norm_cols = df_clean.iloc[:, :normalization_limit]
        df_clean.iloc[:, :normalization_limit] = normalize_dataframe(norm_cols)
        
        # Estandarización
        stnd_cols = df_clean.iloc[:, normalization_limit:]
        df_clean.iloc[:, normalization_limit:] = standardize_dataframe(stnd_cols)
        
        # Reporte
        report = {
            'norm_min': norm_cols.min().tolist(),
            'norm_max': norm_cols.max().tolist(),
            'stnd_mean': stnd_cols.mean().tolist(),
            'stnd_std': stnd_cols.std().tolist()
        }
        return df_clean, report
    
    except Exception as e:
        logging.error(f"Error en clean_dataframe: {str(e)}")
        raise

In [17]:
def load_and_process_data(file_name: str) -> pd.DataFrame:
    """Carga un CSV y elimina la columna 'qW'.
    
    Args:
        file_name (str): Nombre del archivo en RAW_DATA_PATH.
        
    Returns:
        pd.DataFrame: DataFrame procesado.
    """
    file_path = os.path.join(RAW_DATA_PATH, file_name)
    df = pd.read_csv(file_path)
    df.drop('qW', axis=1, inplace=True, errors='ignore')
    return df

In [18]:
# Carga de datasets
datasets = {
    'perth_100': 'WEC_Perth_100.csv',
    'sydney_100': 'WEC_Sydney_100.csv',
    'perth_49': 'WEC_Perth_49.csv',
    'sydney_49': 'WEC_Sydney_49.csv'
}

loaded_data = {key: load_and_process_data(value) for key, value in datasets.items()}

In [19]:
# Definición de límites
WEC_100_LIMIT = 200
WEC_49_LIMIT = 98

# Limpieza de datos
processed_data = {}
try:
    processed_data['perth_100'], report_perth_100 = clean_dataframe(loaded_data['perth_100'], WEC_100_LIMIT)
    processed_data['sydney_100'], report_sydney_100 = clean_dataframe(loaded_data['sydney_100'], WEC_100_LIMIT)
    processed_data['perth_49'], report_perth_49 = clean_dataframe(loaded_data['perth_49'], WEC_49_LIMIT)
    processed_data['sydney_49'], report_sydney_49 = clean_dataframe(loaded_data['sydney_49'], WEC_49_LIMIT)
except Exception as e:
    logging.critical(f"Procesamiento fallido: {str(e)}")
    raise

# Concatenación de resultados
wec_100 = pd.concat(
    [processed_data['perth_100'], processed_data['sydney_100']], 
    ignore_index=True
)
wec_49 = pd.concat(
    [processed_data['perth_49'], processed_data['sydney_49']], 
    ignore_index=True
)

In [20]:
wec_100.to_csv(os.path.join(DATA_PATH, 'wec_100.csv'), index=False)
wec_49.to_csv(os.path.join(DATA_PATH, 'wec_49.csv'), index=False)

# Guardado de reportes
with open(os.path.join(REPORTS_PATH, 'data_stats.json'), 'w') as f:
    json.dump({
        'perth_100': report_perth_100,
        'sydney_100': report_sydney_100,
        'perth_49': report_perth_49,
        'sydney_49': report_sydney_49
    }, f, indent=4)